# Step 11b — Dueling DDQN + SARSA Physician Baseline (Tier 2, GPU)

Trains the Tier 2 FCI-guided RL model on Google Colab GPU.

**Before running:**
1. Upload `rl_dataset_tier2.parquet` to Google Drive at `MyDrive/CareAI/data/rl_dataset_tier2.parquet`
2. Set Runtime → Change runtime type → T4 GPU
3. Run all cells top to bottom

**Outputs saved to:** `MyDrive/CareAI/models/tier2/`

In [ ]:
# ── Cell 1: Mount Drive + check GPU ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found -- running on CPU')

In [ ]:
# ── Cell 2: Install missing packages ─────────────────────────────────────────
# torch, numpy, pandas are pre-installed on Colab; only pyarrow may be missing
!pip install pyarrow --quiet

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
import os

# ---- Paths (edit if you put the file somewhere else in Drive) ----
DRIVE_ROOT  = '/content/drive/MyDrive/CareAI'
DATA_PATH   = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2.parquet')
MODEL_DIR   = os.path.join(DRIVE_ROOT, 'models', 'tier2')
DDQN_DIR    = os.path.join(MODEL_DIR, 'ddqn')
SARSA_DIR   = os.path.join(MODEL_DIR, 'sarsa_phys')

os.makedirs(DDQN_DIR,  exist_ok=True)
os.makedirs(SARSA_DIR, exist_ok=True)

# ---- Hyperparameters ----
N_ACTIONS        = 16       # 2^4 Tier 2 action space
HIDDEN           = 128
LR               = 1e-4
GAMMA            = 0.99
TAU              = 0.001
BATCH_SIZE       = 512      # larger batch for GPU efficiency (was 32 on CPU)
DQN_STEPS        = 100_000
SARSA_STEPS      = 80_000
REWARD_THRESHOLD = 20
REG_LAMBDA       = 5.0
PER_ALPHA        = 0.6
PER_EPSILON      = 0.01
BETA_START       = 0.9
CHECKPOINT_EVERY = 10_000   # more frequent than local (fast on GPU)
LOG_EVERY        = 2_000

print('Config OK')
print(f'  Data:      {DATA_PATH}')
print(f'  Model dir: {MODEL_DIR}')
print(f'  Device:    {device}')
print(f'  Batch:     {BATCH_SIZE}')

In [ ]:
# ── Cell 4: Network architecture ─────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class DuelingDQN(nn.Module):
    """Dueling Double DQN with batch normalisation.
    Architecture: Input -> FC(hidden)->BN->LeakyReLU x2
                  -> Advantage head + Value head -> Q = V + (A - mean(A))
    """
    def __init__(self, n_input, n_actions=16, hidden=128, leaky_slope=0.01):
        super().__init__()
        self.fc1     = nn.Linear(n_input, hidden)
        self.bn1     = nn.BatchNorm1d(hidden)
        self.fc2     = nn.Linear(hidden, hidden)
        self.bn2     = nn.BatchNorm1d(hidden)
        self.adv_fc  = nn.Linear(hidden, 64)
        self.adv_out = nn.Linear(64, n_actions)
        self.val_fc  = nn.Linear(hidden, 64)
        self.val_out = nn.Linear(64, 1)
        self.leaky_slope = leaky_slope

    def forward(self, x):
        x   = F.leaky_relu(self.bn1(self.fc1(x)), self.leaky_slope)
        x   = F.leaky_relu(self.bn2(self.fc2(x)), self.leaky_slope)
        adv = F.leaky_relu(self.adv_fc(x), self.leaky_slope)
        adv = self.adv_out(adv)
        val = F.leaky_relu(self.val_fc(x), self.leaky_slope)
        val = self.val_out(val)
        return val + adv - adv.mean(dim=1, keepdim=True)

print('DuelingDQN defined')

In [ ]:
# ── Cell 5: Prioritized Replay Buffer ────────────────────────────────────────
import numpy as np
from collections import namedtuple

Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, epsilon=0.01):
        self.capacity  = capacity
        self.alpha     = alpha
        self.epsilon   = epsilon
        self.buffer    = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.pos  = 0
        self.size = 0

    def add(self, transition, priority=None):
        if priority is None:
            priority = self.priorities[:self.size].max() if self.size > 0 else 1.0
        if self.size < self.capacity:
            self.buffer.append(transition)
            self.size += 1
        else:
            self.buffer[self.pos] = transition
        self.priorities[self.pos] = priority
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.9):
        probs   = self.priorities[:self.size] ** self.alpha
        probs  /= probs.sum()
        indices = np.random.choice(self.size, batch_size, p=probs, replace=False)
        samples = [self.buffer[i] for i in indices]
        weights = (self.size * probs[indices]) ** (-beta)
        weights /= weights.max()
        return samples, indices, torch.FloatTensor(weights)

    def update_priorities(self, indices, td_errors):
        for idx, td in zip(indices, td_errors):
            self.priorities[idx] = (abs(td) + self.epsilon) ** self.alpha

print('PrioritizedReplayBuffer defined')

In [ ]:
# ── Cell 6: GPU-optimised training functions ──────────────────────────────────
#
# Key differences from the CPU version:
#   1. All data pre-loaded to GPU tensors once (no per-step CPU->GPU transfer)
#   2. Priorities kept as GPU tensor; sampled with torch.multinomial (GPU)
#   3. replacement=True in sampling -- at 512/1M the duplicate rate is 0.05%,
#      statistically identical to without-replacement but O(k) not O(n)
#   4. loss stored as GPU tensor; single transfer to CPU at end (not 100k syncs)
#   5. Priority update is a vectorised tensor assignment, no Python loop
#
import pickle
import time
import torch.optim as optim


def train_dqn(data, n_state, n_actions=16, hidden=128, leaky_slope=0.01,
              lr=1e-4, gamma=0.99, tau=0.001, batch_size=512, num_steps=100000,
              reward_threshold=20, reg_lambda=5.0,
              per_alpha=0.6, per_epsilon=0.01, beta_start=0.9,
              save_dir=None, checkpoint_every=10000,
              device='cpu', log_every=2000):

    print(f'DDQN: {num_steps} steps | n_state={n_state} | n_actions={n_actions} | '
          f'lr={lr:.0e} | batch={batch_size} | device={device}')

    main_net   = DuelingDQN(n_state, n_actions, hidden, leaky_slope).to(device)
    target_net = DuelingDQN(n_state, n_actions, hidden, leaky_slope).to(device)
    target_net.load_state_dict(main_net.state_dict())
    target_net.eval()
    optimizer = optim.Adam(main_net.parameters(), lr=lr)

    # Pre-load entire dataset to GPU once
    print('  Loading data to GPU...')
    n             = len(data['states'])
    states_t      = torch.FloatTensor(data['states']).to(device)
    next_states_t = torch.FloatTensor(data['next_states']).to(device)
    actions_t     = torch.LongTensor(data['actions']).to(device)
    rewards_t     = torch.FloatTensor(data['rewards']).to(device)
    done_t        = torch.FloatTensor(data['done']).to(device)

    # Priority tensor on GPU
    priorities = (torch.FloatTensor(np.abs(data['rewards']) + per_epsilon) ** per_alpha).to(device)
    print(f'  {n:,} transitions on {device}')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    # Store losses on GPU to avoid 100k CPU syncs; transfer once at end
    losses_gpu = torch.zeros(num_steps, device=device)
    t0 = time.time()

    for step in range(num_steps):
        main_net.train()

        # GPU-native sampling
        probs      = priorities / priorities.sum()
        indices    = torch.multinomial(probs, batch_size, replacement=True)
        is_weights = (n * probs[indices]) ** (-beta_start)
        is_weights = (is_weights / is_weights.max()).detach()

        # Batch extraction — pure GPU tensor indexing, no Python loop
        states_b      = states_t[indices]
        next_states_b = next_states_t[indices]
        actions_b     = actions_t[indices]
        rewards_b     = rewards_t[indices]
        done_b        = done_t[indices]

        q_values = main_net(states_b)
        q_taken  = q_values.gather(1, actions_b.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_main   = main_net(next_states_b)
            next_actions  = next_q_main.argmax(dim=1)
            next_q_target = target_net(next_states_b)
            next_q        = next_q_target.gather(1, next_actions.unsqueeze(1)).squeeze(1)
            next_q        = next_q.clamp(-reward_threshold, reward_threshold)
            target        = rewards_b + gamma * next_q * (1 - done_b)

        td_error = target - q_taken
        loss  = (is_weights * td_error.pow(2)).mean()
        reg   = torch.clamp(q_values.abs() - reward_threshold, min=0).sum()
        loss += reg_lambda * reg / batch_size

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(main_net.parameters(), 10.0)
        optimizer.step()

        # Vectorised priority update — no Python loop
        with torch.no_grad():
            priorities[indices] = (td_error.detach().abs() + per_epsilon) ** per_alpha

        # Soft target update
        for p_m, p_t in zip(main_net.parameters(), target_net.parameters()):
            p_t.data.copy_(tau * p_m.data + (1 - tau) * p_t.data)

        # Store loss on GPU — no sync
        losses_gpu[step] = loss.detach()

        if (step + 1) % log_every == 0:
            elapsed   = time.time() - t0
            steps_sec = (step + 1) / elapsed
            eta       = (num_steps - step - 1) / steps_sec
            avg_loss  = losses_gpu[step+1-log_every:step+1].mean().item()  # one sync per log
            pct       = 100 * (step + 1) / num_steps
            print(f'  DDQN {step+1}/{num_steps} ({pct:.0f}%) | '
                  f'loss={avg_loss:.4f} | {steps_sec:.1f} steps/s | '
                  f'elapsed {elapsed:.0f}s | ETA {eta:.0f}s')

        if save_dir and checkpoint_every and (step + 1) % checkpoint_every == 0:
            ckpt = os.path.join(save_dir, f'checkpoint_{step+1}.pt')
            torch.save(main_net.state_dict(), ckpt)
            print(f'  Checkpoint saved: {ckpt}')

    # Final Q-values using already-loaded GPU tensors
    main_net.eval()
    with torch.no_grad():
        all_q = []
        for i in range(0, n, 4096):
            all_q.append(main_net(states_t[i:i+4096]).cpu().numpy())
        all_q       = np.concatenate(all_q, axis=0)
        all_actions = all_q.argmax(axis=1)

    losses = losses_gpu.cpu().numpy().tolist()  # single transfer
    print(f'DDQN complete. Mean Q={all_q.mean():.4f}')

    if save_dir:
        torch.save(main_net.state_dict(), os.path.join(save_dir, 'dqn_model.pt'))
        with open(os.path.join(save_dir, 'dqn_actions.pkl'),  'wb') as f: pickle.dump(all_actions, f)
        with open(os.path.join(save_dir, 'dqn_q_values.pkl'), 'wb') as f: pickle.dump(all_q, f)
        with open(os.path.join(save_dir, 'dqn_losses.pkl'),   'wb') as f: pickle.dump(losses, f)
        print(f'  Saved to {save_dir}')

    return main_net, all_actions, all_q


def compute_q_values(model, states_np, device='cpu', batch_size=4096):
    model.eval()
    all_q = []
    states_t = torch.FloatTensor(states_np).to(device)
    with torch.no_grad():
        for i in range(0, len(states_t), batch_size):
            all_q.append(model(states_t[i:i+batch_size]).cpu().numpy())
    return np.concatenate(all_q, axis=0), np.concatenate(all_q, axis=0).argmax(axis=1)


def train_sarsa_physician(data, n_state, n_actions=16, hidden=128,
                          lr=1e-4, gamma=0.99, tau=0.001, batch_size=512,
                          num_steps=80000, reward_threshold=20, reg_lambda=5.0,
                          per_alpha=0.6, per_epsilon=0.01, beta_start=0.9,
                          save_dir=None, checkpoint_every=10000,
                          device='cpu', log_every=2000):

    print(f'SARSA physician: {num_steps} steps | n_state={n_state} | n_actions={n_actions}')

    main_net   = DuelingDQN(n_state, n_actions, hidden, 0.01).to(device)
    target_net = DuelingDQN(n_state, n_actions, hidden, 0.01).to(device)
    target_net.load_state_dict(main_net.state_dict())
    target_net.eval()
    optimizer = optim.Adam(main_net.parameters(), lr=lr)

    # Pre-load to GPU
    print('  Loading data to GPU...')
    n             = len(data['states'])
    states_t      = torch.FloatTensor(data['states']).to(device)
    next_states_t = torch.FloatTensor(data['next_states']).to(device)
    actions_t     = torch.LongTensor(data['actions']).to(device)
    rewards_t     = torch.FloatTensor(data['rewards']).to(device)
    done_t        = torch.FloatTensor(data['done']).to(device)

    # Physician next-action array — also on GPU
    next_actions_arr = np.zeros(n, dtype=np.int64)
    for i in range(n - 1):
        if data['done'][i] == 0:
            next_actions_arr[i] = data['actions'][i + 1]
    next_actions_t = torch.LongTensor(next_actions_arr).to(device)

    priorities = (torch.FloatTensor(np.abs(data['rewards']) + per_epsilon) ** per_alpha).to(device)
    print(f'  {n:,} transitions on {device}')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    losses_gpu = torch.zeros(num_steps, device=device)
    t0 = time.time()

    for step in range(num_steps):
        main_net.train()

        probs      = priorities / priorities.sum()
        indices    = torch.multinomial(probs, batch_size, replacement=True)
        is_weights = (n * probs[indices]) ** (-beta_start)
        is_weights = (is_weights / is_weights.max()).detach()

        states_b      = states_t[indices]
        next_states_b = next_states_t[indices]
        actions_b     = actions_t[indices]
        rewards_b     = rewards_t[indices]
        done_b        = done_t[indices]

        q_values = main_net(states_b)
        q_taken  = q_values.gather(1, actions_b.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q            = target_net(next_states_b)
            phys_next_actions = next_actions_t[indices]   # GPU tensor indexing
            next_q_taken      = next_q.gather(1, phys_next_actions.unsqueeze(1)).squeeze(1)
            next_q_taken      = next_q_taken.clamp(-reward_threshold, reward_threshold)
            target            = rewards_b + gamma * next_q_taken * (1 - done_b)

        td_error = target - q_taken
        loss  = (is_weights * td_error.pow(2)).mean()
        reg   = torch.clamp(q_values.abs() - reward_threshold, min=0).sum()
        loss += reg_lambda * reg / batch_size

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(main_net.parameters(), 10.0)
        optimizer.step()

        with torch.no_grad():
            priorities[indices] = (td_error.detach().abs() + per_epsilon) ** per_alpha

        for p_m, p_t in zip(main_net.parameters(), target_net.parameters()):
            p_t.data.copy_(tau * p_m.data + (1 - tau) * p_t.data)

        losses_gpu[step] = loss.detach()

        if (step + 1) % log_every == 0:
            elapsed   = time.time() - t0
            steps_sec = (step + 1) / elapsed
            eta       = (num_steps - step - 1) / steps_sec
            avg_loss  = losses_gpu[step+1-log_every:step+1].mean().item()
            pct       = 100 * (step + 1) / num_steps
            print(f'  SARSA {step+1}/{num_steps} ({pct:.0f}%) | '
                  f'loss={avg_loss:.4f} | {steps_sec:.1f} steps/s | '
                  f'elapsed {elapsed:.0f}s | ETA {eta:.0f}s')

        if save_dir and checkpoint_every and (step + 1) % checkpoint_every == 0:
            ckpt = os.path.join(save_dir, f'checkpoint_{step+1}.pt')
            torch.save(main_net.state_dict(), ckpt)
            print(f'  Checkpoint saved: {ckpt}')

    main_net.eval()
    with torch.no_grad():
        all_q = []
        for i in range(0, n, 4096):
            all_q.append(main_net(states_t[i:i+4096]).cpu().numpy())
        all_q       = np.concatenate(all_q, axis=0)
        all_actions = all_q.argmax(axis=1)

    losses = losses_gpu.cpu().numpy().tolist()
    print(f'SARSA complete. Mean Q={all_q.mean():.4f}')

    if save_dir:
        torch.save(main_net.state_dict(), os.path.join(save_dir, 'sarsa_phys_model.pt'))
        with open(os.path.join(save_dir, 'phys_actions.pkl'),  'wb') as f: pickle.dump(all_actions, f)
        with open(os.path.join(save_dir, 'phys_q_values.pkl'), 'wb') as f: pickle.dump(all_q, f)
        with open(os.path.join(save_dir, 'sarsa_losses.pkl'),  'wb') as f: pickle.dump(losses, f)
        print(f'  Saved to {save_dir}')

    return main_net, all_actions, all_q


print('GPU-optimised training functions defined')

In [ ]:
# ── Cell 7: Load data ─────────────────────────────────────────────────────────
import pandas as pd

print(f'Loading {DATA_PATH} ...')
df = pd.read_parquet(DATA_PATH)
print(f'  {len(df):,} rows, {len(df.columns)} cols')

state_cols      = [c for c in df.columns if c.startswith('s_') and not c.startswith('s_next_')]
next_state_cols = [c.replace('s_', 's_next_', 1) for c in state_cols]
n_state         = len(state_cols)

print(f'  State features ({n_state}): {state_cols}')
print(f'  Actions: {df["a"].nunique()} unique (expect {N_ACTIONS})')
print(f'  Reward:  mean={df["r"].mean():.3f}, std={df["r"].std():.3f}, '
      f'min={df["r"].min():.1f}, max={df["r"].max():.1f}')
print(f'  Done:    {100*df["done"].mean():.1f}% terminal')

df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'  Splits: train={len(df_train):,}, val={len(df_val):,}, test={len(df_test):,}')
del df

def prepare_rl_data(df, state_cols, next_state_cols):
    return {
        'states':      df[state_cols].values.astype(np.float32),
        'next_states': df[next_state_cols].values.astype(np.float32),
        'actions':     df['a'].values.astype(int),
        'rewards':     df['r'].values.astype(np.float32),
        'done':        df['done'].values.astype(np.float32),
    }

print('Preparing arrays...')
train_data = prepare_rl_data(df_train, state_cols, next_state_cols)
val_data   = prepare_rl_data(df_val,   state_cols, next_state_cols)
test_data  = prepare_rl_data(df_test,  state_cols, next_state_cols)
print('Data ready.')

In [ ]:
# ── Cell 8: SMOKE TEST -- run this before full training ───────────────────────
# 500 steps each, saves to /tmp/smoke_tier2/ (not Drive).
# All assertions must pass before proceeding to the full training cells.

SMOKE_DIR   = '/tmp/smoke_tier2'
SMOKE_STEPS = 500
SMOKE_BATCH = 64    # small but enough to exercise the GPU path
SMOKE_LOG   = 100

print('=' * 60)
print('SMOKE TEST -- 500 steps DDQN + 500 steps SARSA')
print('=' * 60)

# -- DDQN smoke --
sm_dqn, sm_dqn_actions, sm_dqn_q = train_dqn(
    train_data,
    n_state          = n_state,
    n_actions        = N_ACTIONS,
    hidden           = HIDDEN,
    batch_size       = SMOKE_BATCH,
    num_steps        = SMOKE_STEPS,
    save_dir         = os.path.join(SMOKE_DIR, 'ddqn'),
    checkpoint_every = 0,
    device           = device,
    log_every        = SMOKE_LOG,
)

# -- SARSA smoke --
sm_sarsa, sm_sarsa_actions, sm_sarsa_q = train_sarsa_physician(
    train_data,
    n_state          = n_state,
    n_actions        = N_ACTIONS,
    hidden           = HIDDEN,
    batch_size       = SMOKE_BATCH,
    num_steps        = SMOKE_STEPS,
    save_dir         = os.path.join(SMOKE_DIR, 'sarsa_phys'),
    checkpoint_every = 0,
    device           = device,
    log_every        = SMOKE_LOG,
)

# -- Assertions --
errors = []

if n_state != 5:
    errors.append(f'n_state={n_state}, expected 5')
if sm_dqn_q.shape != (len(train_data['states']), N_ACTIONS):
    errors.append(f'DDQN Q shape {sm_dqn_q.shape}, expected ({len(train_data["states"])}, {N_ACTIONS})')
if not np.isfinite(sm_dqn_q).all():
    errors.append('DDQN Q-values contain NaN or Inf')
if sm_dqn_actions.min() < 0 or sm_dqn_actions.max() >= N_ACTIONS:
    errors.append(f'DDQN actions out of range [0, {N_ACTIONS-1}]')
if sm_sarsa_q.shape != (len(train_data['states']), N_ACTIONS):
    errors.append(f'SARSA Q shape wrong')
if not np.isfinite(sm_sarsa_q).all():
    errors.append('SARSA Q-values contain NaN or Inf')

for fname in ['dqn_losses.pkl', 'dqn_model.pt']:
    p = os.path.join(SMOKE_DIR, 'ddqn', fname)
    if not os.path.exists(p):
        errors.append(f'Missing file: {fname}')
for fname in ['sarsa_losses.pkl', 'sarsa_phys_model.pt']:
    p = os.path.join(SMOKE_DIR, 'sarsa_phys', fname)
    if not os.path.exists(p):
        errors.append(f'Missing file: {fname}')

with open(os.path.join(SMOKE_DIR, 'ddqn', 'dqn_losses.pkl'), 'rb') as f:
    smoke_losses = pickle.load(f)
if len(smoke_losses) != SMOKE_STEPS:
    errors.append(f'Loss list length {len(smoke_losses)}, expected {SMOKE_STEPS}')

print()
if errors:
    for e in errors:
        print(f'  FAIL: {e}')
    raise RuntimeError('Smoke test FAILED -- do not proceed to full training')
else:
    print(f'  n_state:          {n_state} (expect 5)')
    print(f'  n_actions:        {N_ACTIONS} (expect 16)')
    print(f'  DDQN Q shape:     {sm_dqn_q.shape}')
    print(f'  DDQN unique acts: {sorted(np.unique(sm_dqn_actions).tolist())}')
    print(f'  DDQN mean Q:      {sm_dqn_q.mean():.4f}')
    print(f'  SARSA mean Q:     {sm_sarsa_q.mean():.4f}')
    print(f'  Loss list len:    {len(smoke_losses)} (expect {SMOKE_STEPS})')
    print(f'  Device used:      {device}')
    print()
    print('SMOKE TEST PASSED -- safe to proceed to full training')

In [ ]:
# ── Cell 8: Train DDQN ────────────────────────────────────────────────────────
dqn_model, dqn_actions_train, dqn_q_train = train_dqn(
    train_data,
    n_state          = n_state,
    n_actions        = N_ACTIONS,
    hidden           = HIDDEN,
    lr               = LR,
    gamma            = GAMMA,
    tau              = TAU,
    batch_size       = BATCH_SIZE,
    num_steps        = DQN_STEPS,
    reward_threshold = REWARD_THRESHOLD,
    reg_lambda       = REG_LAMBDA,
    per_alpha        = PER_ALPHA,
    per_epsilon      = PER_EPSILON,
    beta_start       = BETA_START,
    save_dir         = DDQN_DIR,
    checkpoint_every = CHECKPOINT_EVERY,
    device           = device,
    log_every        = LOG_EVERY,
)

# Save val/test Q-values
for split_name, split_data in [('val', val_data), ('test', test_data)]:
    q_vals, actions = compute_q_values(dqn_model, split_data['states'], device=device)
    with open(os.path.join(DDQN_DIR, f'dqn_actions_{split_name}.pkl'), 'wb') as f: pickle.dump(actions, f)
    with open(os.path.join(DDQN_DIR, f'dqn_q_{split_name}.pkl'),       'wb') as f: pickle.dump(q_vals, f)
    print(f'  DQN {split_name}: mean Q={q_vals.mean():.4f}, {len(np.unique(actions))} unique actions')

In [ ]:
# ── Cell 9: Train SARSA physician baseline ────────────────────────────────────
sarsa_model, sarsa_actions_train, sarsa_q_train = train_sarsa_physician(
    train_data,
    n_state          = n_state,
    n_actions        = N_ACTIONS,
    hidden           = HIDDEN,
    lr               = LR,
    gamma            = GAMMA,
    tau              = TAU,
    batch_size       = BATCH_SIZE,
    num_steps        = SARSA_STEPS,
    reward_threshold = REWARD_THRESHOLD,
    reg_lambda       = REG_LAMBDA,
    per_alpha        = PER_ALPHA,
    per_epsilon      = PER_EPSILON,
    beta_start       = BETA_START,
    save_dir         = SARSA_DIR,
    checkpoint_every = CHECKPOINT_EVERY,
    device           = device,
    log_every        = LOG_EVERY,
)

# Save val/test Q-values
for split_name, split_data in [('val', val_data), ('test', test_data)]:
    q_vals, actions = compute_q_values(sarsa_model, split_data['states'], device=device)
    with open(os.path.join(SARSA_DIR, f'phys_actions_{split_name}.pkl'), 'wb') as f: pickle.dump(actions, f)
    with open(os.path.join(SARSA_DIR, f'phys_q_{split_name}.pkl'),       'wb') as f: pickle.dump(q_vals, f)
    print(f'  SARSA {split_name}: mean Q={q_vals.mean():.4f}, {len(np.unique(actions))} unique actions')

In [ ]:
# ── Cell 10: Verify outputs ───────────────────────────────────────────────────
print('Files saved to Drive:')
for root, dirs, files in os.walk(MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, MODEL_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nDone. Download models/tier2/ from Drive and place in:')
print('  CareAI/models/icu_readmit/tier2/')